In [1]:
import json
import re
from datasets import load_dataset

## DDI 2013

In [ ]:
import os
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))

# Login using e.g. `huggingface-cli login` to access this dataset
# ds = load_dataset("clinicalnlplab/DDI2013_train")
ds = load_dataset("clinicalnlplab/DDI2013_test")

train-00000-of-00001.parquet:   0%|          | 0.00/3.88M [00:00<?, ?B/s]

valid-00000-of-00001.parquet:   0%|          | 0.00/1.47M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18779 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/7244 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5761 [00:00<?, ? examples/s]

In [44]:
for key, val in ds.items():
    print(f'{key}: {len(val)}')

train: 18779
valid: 7244
test: 5761


In [45]:
## example
ds['train'][0]

{'id': 'DDI20130',
 'query': "TASK: The task is to predict relationship between the given head entity labeled as @DRUG1$ and tail entity labeled as @DRUG2$ within a given sentence, this relation which must be in (‘mechanism’, ‘effect’, ‘advice’, ‘int’, 'none’).\nmechanism: this type is use to annotate drug-drug interactions that are described by their pharmacokinetic mechanism.\neffect: this type is used to annotate drug-drug interactions describing an effect or a pharmacodynamic mechanism.\nadvice: this type is used when a recommendation or advide regarding a drug interaction is given.\nint: this type is used when a drug-durg interactions apprears in the text without providing any additional information.\nnone: there has no drug-drug interactions.\nINPUT: Both efavirenz and nevirapine have been compared to triple therapy with the @DRUG1$ indinavir over 48 weeks as initial therapy, with similar responses being observed with @DRUG2$ regimens and superiority observed with efavirenz.\nWha

In [46]:
def modify_example(example):
    # if "INPUT:" in example["query"]:
    #     example["query"] = example["query"].split("INPUT: ", 1)[1].split("\nWhat is the relationship between")[0].strip()
    # return example
    try:
        pattern = re.compile(r'TASK:(.*?)INPUT:(.*?)OUTPUT:(.*)', re.DOTALL)
        match = pattern.search(example['query'])
        task_part = match.group(1).strip()
        task_desc = "Please do a relation extraction task. You need to extract the relationship between the given head entity and tail entity. The response should be in predefined set: {'mechanism', 'effect, 'advice', 'int', 'none'}."
        task_lines = [line.strip() for line in task_part.split('\n') if line.strip()]
        task_part = '\n'.join([task_desc] + task_lines[1:])
        input_part = match.group(2).strip()

    except:
        print(example)

    return {
        'instruction': task_part,
        'inputsentence': input_part,
        'response': example['answer'],
    }

In [ ]:
modes = ['train', 'test']

for mode in modes:
    output_file = f'RE_DDI_{mode}.json'
    with open(output_file, 'w') as fout:
        for example in ds[mode]:
            print(json.dumps(modify_example(example)), file=fout)

## ChemProt

In [2]:
import pandas as pd
import nltk
from nltk.tokenize import PunktSentenceTokenizer

nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/wan02633/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
train_df = pd.read_parquet("../mmrag/datasets/ChemProt_train.parquet")
test_df = pd.read_parquet("../mmrag/datasets/ChemProt_test.parquet")

In [ ]:
print(len(train_df), len(test_df))

1020 612 800


In [6]:
def extract_sentences(text, nouns, offsets):
    # 初始化句子分割器并获取带位置信息的句子
    tokenizer = PunktSentenceTokenizer()
    sentence_spans = list(tokenizer.span_tokenize(text))
    # print(sentence_spans)

    sent_list, entity_list = [], []
    for sent_start, sent_end in sentence_spans:
        sentence = text[sent_start: sent_end].strip()
        sent_list.append(sentence)

        entities = []
        for noun, offset in zip(nouns, offsets):
            start_pos, end_pos = offset
            if sent_start <= start_pos < sent_end and text[start_pos: end_pos] not in entities:
                entities.append(text[start_pos: end_pos])
        entity_list.append(entities)
    return sent_list, entity_list
    
    # result = {}
    # for noun, offset in zip(nouns, offsets):
    #     start_pos = offset[0]
    #     # 遍历所有句子范围寻找匹配
    #     for sent_start, sent_end in sentence_spans:
    #         if sent_start <= start_pos < sent_end:
    #             result[noun] = text[sent_start:sent_end].strip()
    #             break
    # return result

In [7]:
def modify_dataframe(df, output_file):
    task_desc = "Please do a named entity recognition task. You need to to accurately recognize chemical or genetic words or terms given the input sentece."
    task_desc += " The response should only be entities following the form shown in original sentences."
    task_desc += " If the given sentence does not include these kinds of entities, just outpout None."
    fout = open(output_file, 'w')
    
    all_types = set()
    for idx, row in df.iterrows():
        # print(row.text)
        # print(row.entities.keys())
        for t in row.entities['type']:
            all_types.add(t)
    
        text = row.text
        nouns = row.entities['text']
        offsets = row.entities['offsets']
        assert len(nouns) == len(offsets)
        for noun, (start_idx, end_idx) in zip(nouns, offsets):
            assert text[start_idx: end_idx] == noun
        
        sent_list, entity_list = extract_sentences(text, nouns, offsets)
        for sent, entities in zip(sent_list, entity_list):
            # print(sent, entities)
            if len(sent) < 20 and len(entities) == 0:
                continue
            entities = None if len(entities) == 0 else entities
            res = {
                    'instruction': task_desc,
                    'inputsentence': sent,
                    'response': str(entities),
                }
            print(json.dumps(res), file=fout)

    print(all_types)
    fout.close()

In [8]:
num = 0
tokenizer = PunktSentenceTokenizer()
for idx, row in train_df.iterrows():
    sent_spans = list(tokenizer.span_tokenize(row.text))
    num += len(sent_spans)
print(num)

10466


In [ ]:
modify_dataframe(train_df, 'NER_chemprot_train.json')
modify_dataframe(test_df, 'NER_chemprot_test.json')